# Continuous polynomial macrofaces in three-dimensional MHM

The primal tetrahedral Darcy formulation supports discontinuous or continuous piecewise polynomial normal-flux densities within each triangular macroface. `continuous=True` shares Bernstein coefficients on internal subtriangle edges and vertices. Distinct macrofaces retain separate coordinates. A boolean sequence mixes the choices per face, and a degree sequence assigns independent polynomial degrees. Every positive degree is supported for continuous faces; discontinuous faces also accept degree zero.

This notebook uses the generic local/global formulation helper to solve an affine pressure patch. It compares DG P1, C0 P1, mixed DG/C0 faces, and C0 faces with different positive degrees. The exact data are $p=1+x+2y+3z$, identity permeability, zero source, nonhomogeneous Dirichlet pressure, and physical Darcy flux $q=(-1,-2,-3)$. Polynomial local pressure spaces resolve this patch. Fine boundaries must resolve each face partition, and the local trace coupling must have adequate rank.

The [face-based MHM paper by Paredes, Valentin and Versieux (2024)](https://doi.org/10.1016/j.cam.2023.115415) covers dimensions two and three. Its [author preprint](https://www.ci2ma.udec.cl/pdf/pre-publicaciones/2022/pp22-31.pdf), §3.2, equation (22), defines continuity within independently partitioned macrofaces. The published numerical studies are two-dimensional. This three-dimensional example is an original analytical implementation check; it does not reproduce a published benchmark or establish uniform stability for arbitrary finite local spaces.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/5cefcf3fa1c1cd3eb2736aaa7e9ccdaccbf23ac4ffd2c0fab00bf774d62a60d9/74_continuous_macrofaces3d-companion.zip"
COMPANION_SHA256 = "5cefcf3fa1c1cd3eb2736aaa7e9ccdaccbf23ac4ffd2c0fab00bf774d62a60d9"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/74_continuous_macrofaces3d.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


In [ ]:
import numpy as np
from pymhm import assemble
from pymhm.meshes.tetrahedron import TetraMesh
from pymhm.fem.traces.triangle_3d import TriangularSkeleton
from examples.formulations.darcy import pressure_constraints
from examples.formulations.tetrahedral_darcy import (
    define_tetrahedral_darcy,
    recover_tetrahedral_darcy,
)


def pressure(points: np.ndarray) -> np.ndarray:
    """Evaluate the prescribed affine pressure patch."""
    return 1 + points @ np.array([1.0, 2.0, 3.0])


mesh = TetraMesh.unit_cube(1)
interior_face_flags = mesh.face_cells[:, 1] >= 0
face_degrees = 1 + np.arange(len(mesh.faces)) % 2
face_choices = (
    ('DG P1', False, 1, 3),
    ('C0 P1', True, 1, 3),
    ('C0 interior / DG boundary', interior_face_flags, 1, 3),
    ('C0 P1/P2 by face', True, face_degrees, 4),
)


In [ ]:
face_patch_rows = []
for label, continuity, face_degree, local_degree in face_choices:
    face_space = TriangularSkeleton(
        mesh, subdivisions=2, degree=face_degree, continuous=continuity
    )
    definition = define_tetrahedral_darcy(
        mesh, skeleton=face_space, degree=local_degree, local_refinement=2,
        dirichlet=pressure,
    )
    system = assemble(definition.problem)
    hybrid = system.solve(constraints=pressure_constraints(definition, system))
    solution = recover_tetrahedral_darcy(definition, system, hybrid)
    pressure_error = solution.l2_error(pressure)
    flux_error = solution.flux_l2_error([-1.0, -2.0, -3.0])
    macro_balance = float(np.max(np.abs(solution.conservation_residuals())))
    assert pressure_error < 1e-10
    assert flux_error < 1e-10
    assert macro_balance < 1e-11
    face_patch_rows.append({
        'space': label,
        'local_degree': local_degree,
        'pressure_l2_error': pressure_error,
        'physical_flux_l2_error': flux_error,
        'macro_balance_max': macro_balance,
        'skeleton_coordinates': face_space.size,
    })
assert (
    face_patch_rows[1]['skeleton_coordinates']
    < face_patch_rows[2]['skeleton_coordinates']
    < face_patch_rows[0]['skeleton_coordinates']
)
face_patch_rows


The reported norms integrate the reconstructed pressure and its physical raw flux over the volume. The integrated macro balance uses the oriented skeleton flux; it does not assert fine-cell conservation or global H(div) conformity of the raw gradient. Skeletal coordinate counts precede boundary treatment and exclude local setup work.

C0 P1 on a face split into four subtriangles has six coordinates, while DG P1 has twelve. These are Bernstein polynomial coefficients, not integrated flux values. Continuity is imposed within each macroface, rather than across different macrofaces or as the separate continuous pressure trace of MH²M.

Canonical RT moment reconstruction can consume these shared face coefficients. The published energy estimator, however, requires independent polynomial tests on each skeletal subface. Subdivided C0 faces remove some of those tests and are excluded by that estimator API. This distinguishes an algebraic flux reconstruction from the hypotheses of its cited error estimates.
